# Machine Learning Engineer Capstone: Moisture Content
**Objective:** Construct an end-to-end machine learning workflow to predict humidity, temperature, and past moisture's effects on moisture content. Is moisture influenced by historical or environmental trends, or do these trends work in tandem?

## Required Libraries

In [1]:
# ... for data science
import numpy as np
import pandas as pd

In [2]:
# ... for machine learning
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, Lasso, Ridge
from sklearn.model_selection import GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.decomposition import PCA
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

## EDA for Linear Regression

In [3]:
df = pd.read_csv('0000 Devens Logger Data.csv', header=1) # because the first row contains group labels while the second contains the actual column names

C:\Windows\Temp\ipykernel_15032\3352695845.py:1: DtypeWarning: Columns (0: Unnamed: 200, 1: N1-MCEM-MC.1, 2: N2-MCEM-MC.1, 3: N3-MCEM-MC.1, 4: Unnamed: 204, 5: Unnamed: 205, 6: S1-MCEM-MC.1, 7: S2-MCEM-MC.1, 8: S3-MCEM-MC.1, 9: Unnamed: 209, 10: Unnamed: 210, 11: Unnamed: 214, 12: Unnamed: 218, 13: Unnamed: 219, 14: N1 T 30 Day Average, 15: N2 T 30 Day Average, 16: N3 T 30 Day Average, 17: Unnamed: 229, 18: Sheath T (30 Day Avg), 19: Unnamed: 234, 20: S1 T 30 Day Average) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('0000 Devens Logger Data.csv', header=1) # because the first row contains group labels while the second contains the actual column names


In [4]:
# Preview the dataset
df.head()

,DateAndTime,T out,RH out,RH out-C,RH out-DP,N1-TCEM,N1-TCNM,N1-TCIM,N1-TENU,N1-TENM,...,S2 T 30 Day Average,S2 Pass-Fail,S3 RH 30 Day Average,S3 T 30 Day Average,S3 Pass-Fail,Unnamed: 244,S1.1,S2.1,S3.1,Sheath T (30 Day Avg).1
0,12/06/2011 20:00,NaN,11.4,11.4,-105.3,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,12/07/2011 15:00,NaN,21.3,21.3,-110.5,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,12/07/2011 16:00,NaN,54.5,54.5,-55.6,10.3,13.7,16.8,8.4,8.4,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,12/07/2011 17:00,NaN,16.0,16.0,-57.4,10.0,13.4,16.7,8.1,8.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,12/07/2011 18:00,NaN,53.1,53.1,-39.8,9.8,13.3,16.8,7.9,7.9,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


**Verdict:** The majority of variables in both subsets are floats, so it makes sense to apply a regression algorithm.

### Data Partition: North and South
Temperature and humidity undoubtedly affect moisture. The question is, how much do these factors differ between north and south exposures?

In [5]:
weather_cols = ['T out', 'RH out', 'RH out-DP']
datetime_col = ['DateAndTime']

In [6]:
df_north = df.filter(regex=r'^N[123]-').copy()
df_north[weather_cols] = df[weather_cols]
df_north[datetime_col] = df[datetime_col]
df_south = df.filter(regex=r'^S[123]-').copy()
df_south[weather_cols] = df[weather_cols]
df_south[datetime_col] = df[datetime_col]

In [7]:
# Data types
print('North side data types:')
print(df_north.dtypes)
print('\nSouth side data types:')
print(df_south.dtypes)

North side data types:
N1-TCEM         float64
N1-TCNM         float64
N1-TCIM         float64
N1-TENU         float64
N1-TENM         float64
                 ...   
N3-MCEM-MC.1     object
T out           float64
RH out          float64
RH out-DP       float64
DateAndTime         str
Length: 92, dtype: object

South side data types:
S1-TCEM         float64
S1-TCNM         float64
S1-TCIM         float64
S1-TENU         float64
S1-TENM         float64
                 ...   
S3-MCEM-MC.1     object
T out           float64
RH out          float64
RH out-DP       float64
DateAndTime         str
Length: 93, dtype: object


In [8]:
# Summary statistics
print('North side summary statistics:')
print(df_north.describe())
print('\nSouth side summary statistics:')
print(df_south.describe())

North side summary statistics:
            N1-TCEM       N1-TCNM       N1-TCIM       N1-TENU       N1-TENM  \
count  22861.000000  22861.000000  22861.000000  22861.000000  22861.000000   
mean      12.097804     16.092734     20.583890     10.689056     10.185434   
std        9.863197      6.373062      3.028832     11.572328     11.830849   
min      -14.100000      0.700000     10.400000    -21.300000    -23.100000   
25%        4.200000     10.900000     18.300000      1.500000      1.100000   
50%       11.400000     15.300000     20.000000     10.000000      9.500000   
75%       20.000000     21.400000     22.800000     19.700000     19.300000   
max       36.700000     32.500000     31.300000     39.300000     42.100000   

            N1-TENL       N1-TDEM       N1-TDIM       N1-TAEM       N2-TCEM  \
count  22861.000000  22861.000000  22861.000000  22861.000000  22861.000000   
mean       9.999077     10.629203     12.148078     21.579612     11.917318   
std       11.686534 

In [9]:
# Missing values
print('North side missing value count:')
print(df_north.isna().sum())
print('\nSouth side missing value count:')
print(df_south.isna().sum())

North side missing value count:
N1-TCEM           18
N1-TCNM           18
N1-TCIM           18
N1-TENU           18
N1-TENM           18
                ... 
N3-MCEM-MC.1    1917
T out             39
RH out            16
RH out-DP         16
DateAndTime       14
Length: 92, dtype: int64

South side missing value count:
S1-TCEM           18
S1-TCNM           18
S1-TCIM           18
S1-TENU           18
S1-TENM           18
                ... 
S3-MCEM-MC.1    5386
T out             39
RH out            16
RH out-DP         16
DateAndTime       14
Length: 93, dtype: int64


## Feature Engineering

### Data Cleaning
* `DateAndTime` needs to be converted from strings to datetimes.
* `N1-MCEM_MC.1` to `S3-MCEM-MC.1` can be removed, considering they're duplicates.
* Later on, missing target values need to be removed, as well as associated feature values.

In [10]:
# Datetime conversion (north)
raw_dates = df_north['DateAndTime'].astype('string').str.strip() # parses timestamp formats explicitly
dates = pd.to_datetime(raw_dates, format='%m/%d/%Y %H:%M', errors='coerce')
dates = dates.fillna(pd.to_datetime(raw_dates, format='%m/%d/%y %H:%M', errors='coerce'))
invalid_dates = dates.isna()
df_north = df_north.loc[~invalid_dates].copy() # excludes rows without usable timestamps
df_north['DateAndTime'] = dates.loc[~invalid_dates]
df_north = df_north.sort_values('DateAndTime')

In [11]:
# Datetime conversion (south)
raw_dates = df_south['DateAndTime'].astype('string').str.strip()
dates = pd.to_datetime(raw_dates, format='%m/%d/%Y %H:%M', errors='coerce')
dates = dates.fillna(pd.to_datetime(raw_dates, format='%m/%d/%y %H:%M', errors='coerce'))
invalid_dates = dates.isna()
df_south = df_south.loc[~invalid_dates].copy()
df_south['DateAndTime'] = dates.loc[~invalid_dates]
df_south = df_south.sort_values('DateAndTime')

In [12]:
# Duplicate column removals
duplicate_regex = r'^[NS][123]-MCEM-MC\.1$'
df_north = df_north.drop(columns=df_north.filter(regex=duplicate_regex).columns)
df_south = df_south.drop(columns=df_south.filter(regex=duplicate_regex).columns)

In [13]:
print('North side preview:')
print(df_north.head())
print('South side preview:')
print(df_south.head())

North side preview:
   N1-TCEM  N1-TCNM  N1-TCIM  N1-TENU  N1-TENM  N1-TENL  N1-TDEM  N1-TDIM  \
0      NaN      NaN      NaN      NaN      NaN      NaN      NaN      NaN   
1      NaN      NaN      NaN      NaN      NaN      NaN      NaN      NaN   
2     10.3     13.7     16.8      8.4      8.4      8.2      9.2     10.9   
3     10.0     13.4     16.7      8.1      8.0      7.9      8.8     10.6   
4      9.8     13.3     16.8      7.9      7.9      7.8      8.7     10.4   

   N1-TAEM  N2-TCEM  ...  N3-RCEM  N3-RCEM-C  N3-RCEM-DP  N3-RAIM  N3-RAIM-C  \
0      NaN      NaN  ...      7.4        NaN         NaN      5.1        NaN   
1      NaN      NaN  ...      NaN        NaN         NaN      NaN        NaN   
2     17.3     10.7  ...     85.6       83.1         8.5     59.3       58.2   
3     17.4     10.4  ...     85.6       83.0         8.1     59.5       58.4   
4     17.5     10.2  ...     85.3       82.7         7.9     59.6       58.5   

   N3-RAIM-DP  T out  RH out  RH out

### Preprocessing Pipeline (to be used later)
* Fills in missing values
* Scales columns
* Reduces dimensionality

In [14]:
preprocessor = Pipeline([('imputer', SimpleImputer(strategy='mean')),
                         ('scale', StandardScaler()),
                         ('pca', PCA(n_components=0.95))])

### Features and Targets
* `X_hist`: past moisture & average moisture
* `X_env`: temperature & humidity
* `y`: expected moisture

In [15]:
moisture_regex = r'^[NS][123]-M.*-MC$'
temp_regex = r'^[NS][123]-[T]'
hum_regex = r'^[NS][123]-[R]'

#### North Exposure

In [16]:
# Environmental factors
y_north = df_north.filter(regex=moisture_regex).copy()
X_temp_north = df_north.filter(regex=temp_regex).copy()
X_hum_north = df_north.filter(regex=hum_regex).copy()
X_weather_north = df_north[weather_cols]
X_env_north = pd.concat([X_temp_north, X_hum_north, X_weather_north], axis=1)

In [17]:
# Moisture history
north_lag = y_north.shift(1) # the previous observation
north_prev_mean = north_lag.rolling(24).mean() # the average of the past 24 observations
X_north_lag, X_north_previous_mean = north_lag.add_suffix('_lag_1'), north_prev_mean.add_suffix('_mean_previous_24')
X_hist_north = pd.concat([X_north_lag, X_north_previous_mean], axis=1)

In [18]:
# Missing target removal
valid = y_north.notna().all(axis=1)
X_env_north = X_env_north.loc[valid].copy()
X_hist_north = X_hist_north.loc[valid].copy()
y_north = y_north.loc[valid].copy()

#### South Exposure

In [19]:
# Environmental factors
y_south = df_south.filter(regex=moisture_regex).copy()
X_temp_south = df_south.filter(regex=temp_regex).copy()
X_hum_south = df_south.filter(regex=hum_regex).copy()
X_weather_south = df_south[weather_cols]
X_env_south = pd.concat([X_temp_south, X_hum_south, X_weather_south], axis=1)

In [20]:
# Moisture history
south_lag = y_south.shift(1) # the previous observation
south_previous_mean = south_lag.rolling(24).mean() # the average of the past 24 observations
X_south_lag, X_south_previous_mean = south_lag.add_suffix('_lag_1'), south_previous_mean.add_suffix('_mean_previous_24')
X_hist_south = pd.concat([X_south_lag, X_south_previous_mean], axis=1)

In [21]:
# Missing target removal
valid = y_south.notna().all(axis=1)
X_env_south = X_env_south.loc[valid].copy()
X_hist_south = X_hist_south.loc[valid].copy()
y_south = y_south.loc[valid].copy()

## Implement Machine Learning Workflows
**Model Inputs**
* Features
    * Environmental factors (ie temperature, humidity)
    * Historical factors (ie prior moisture content)
* Targets
    * `y_north`
    * `y_south`

### Baseline: Current Moisture v lag
**NB** Direct persistence will be used instead of regression or PCA. This way, we will be able to see whether the moisture data behaves like a random walk in which current moisture content readings predict future readings.

#### North Exposure

In [22]:
# Align previous measurements with retained targets
lag_aligned = north_lag.loc[y_north.index].copy()

In [23]:
# Same chronological 80/20 split that will be used by future models
split_at = int(len(y_north)*0.80)
y_test = y_north.iloc[split_at:]
y_pred = lag_aligned.iloc[split_at:]

In [24]:
# Persistence needs an available previous measurement
valid = y_test.notna().all(axis=1) & y_pred.notna().all(axis=1)
y_test = y_test.loc[valid]
y_pred = y_pred.loc[valid]

In [25]:
# Evaluate directly: no fit(), preprocessing, or regression
mae = mean_absolute_error(y_test, y_pred)
print(f'Persistence MAE: {mae:.3f}')
mse = mean_squared_error(y_test, y_pred)
print(f'Persistence MSE: {mse:.3f}')
r2 = r2_score(y_test, y_pred)
print(f'Persistence R²: {r2:.3f}')

Persistence MAE: 0.062
Persistence MSE: 0.015
Persistence R²: 0.994


#### South Exposure

In [26]:
# Align previous measurements with retained targets
lag_aligned = south_lag.loc[y_south.index].copy()

In [27]:
# Same chronological 80/20 split that will be used by future models
split_at = int(len(y_south)*0.80)
y_test = y_south.iloc[split_at:]
y_pred = lag_aligned.iloc[split_at:]

In [28]:
# Persistence needs an available previous measurement
valid = y_test.notna().all(axis=1) & y_pred.notna().all(axis=1)
y_test = y_test.loc[valid]
y_pred = y_pred.loc[valid]

In [99]:
# Evaluate directly: no fit(), preprocessing, or regression
mae = mean_absolute_error(y_test, y_pred)
print(f'Persistence MAE: {mae:.3f}')
mse = mean_squared_error(y_test, y_pred)
print(f'Persistence MSE: {mse:.3f}')
r2 = r2_score(y_test, y_pred)
print(f'Persistence R²: {r2:.3f}')

Persistence MAE: 0.111
Persistence MSE: 0.026
Persistence R²: 0.973


**Verdict:** Because errors are higher on the south side of the building than the north side, successive south-side moisture measurements cannot be predicted as easily. This does not, however, suggest the south side is wetter, has greater damage risk, or is more affected by temperature and humidity.

### History

In [30]:
lr_hist = LinearRegression()

#### North Exposure

In [31]:
# The training-testing split
X_train, X_test, y_train, y_test = train_test_split(X_hist_north,
                                                    y_north,
                                                    random_state=42,
                                                    test_size=0.20,
                                                    shuffle=False)

In [32]:
# Data cleaning
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

In [33]:
# Model fitting
lr_hist.fit(X_train_processed, y_train)

,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies the convergence criterion of the underlying solver. `tol` isset as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. `tol` is set as `cond` of:func:`scipy.linalg.lstsq` when fitting on dense training data... versionadded:: 1.7.. versionchanged:: 1.9 Now supported on dense data, interpreted as the `cond` parameter.",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary <n_jobs>` for more details.",None
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive. Thisoption is only supported for dense arrays.For a comparison between a linear regression model with positive constraintson the regression coefficients and a linear regression without such constraints,see :ref:`sphx_glr_auto_examples_linear_model_plot_nnls.py`... versionadded:: 0.24",False
Name,Type,Value
"coef_ coef_: array of shape (n_features, ) or (n_targets, n_features)Estimated coefficients for the linear regression problem.If multiple targets are passed during the fit (y 2D), thisis a 2D array of shape (n_targets, n_features), while if onlyone target is passed, this is a 1D array of length n_features.","ndarray[float64](18, 3)","[[ 0.41,-0.18, 0. ], [ 0.69,-0.68,-0.06], [ 0.54,-0.24, 0.23], ..., [ 0.44, 0.54, 0.03], [ 0.25, 0.01, 0.83], [ 1.34,-0.81, 0.74]]"
"intercept_ intercept_: float or array of shape (n_targets,)Independent term in the linear model. Set to 0.0 if`fit_intercept = False`.","ndarray[float64](18,)","[13.29,15.16,14.37,...,12.54,10.76,17.88]"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`... versionadded:: 0.24,int,3
rank_ rank_: intRank of matrix `X`. Only available when `X` is dense.,int64,np.int64(3)
"singular_ singular_: array of shape (min(X, y),)Singular values of `X`. Only available when `X` is dense.","ndarray[float64](3,)","[758.38,135.46,123.38]"


In [34]:
# Model accuracy
y_pred = lr_hist.predict(X_test_processed)
mae = mean_absolute_error(y_test, y_pred)
print(f'North MAE: {mae:.3f}')
mse = mean_squared_error(y_test, y_pred)
print(f'North MSE: {mse:.3f}')
r2 = r2_score(y_test, y_pred)
print(f'North R²: {r2:.3f}')

North MAE: 0.592
North MSE: 0.754
North R²: 0.802


#### South Exposure

In [35]:
# The training-testing split
X_train, X_test, y_train, y_test = train_test_split(X_hist_south,
                                                    y_south,
                                                    random_state=42,
                                                    test_size=0.20,
                                                    shuffle=False)

In [36]:
# Data cleaning
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

In [37]:
# Model fitting
lr_hist.fit(X_train_processed, y_train)

,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies the convergence criterion of the underlying solver. `tol` isset as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. `tol` is set as `cond` of:func:`scipy.linalg.lstsq` when fitting on dense training data... versionadded:: 1.7.. versionchanged:: 1.9 Now supported on dense data, interpreted as the `cond` parameter.",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary <n_jobs>` for more details.",None
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive. Thisoption is only supported for dense arrays.For a comparison between a linear regression model with positive constraintson the regression coefficients and a linear regression without such constraints,see :ref:`sphx_glr_auto_examples_linear_model_plot_nnls.py`... versionadded:: 0.24",False
Name,Type,Value
"coef_ coef_: array of shape (n_features, ) or (n_targets, n_features)Estimated coefficients for the linear regression problem.If multiple targets are passed during the fit (y 2D), thisis a 2D array of shape (n_targets, n_features), while if onlyone target is passed, this is a 1D array of length n_features.","ndarray[float64](18, 3)","[[ 0.36,-0.09, 0.24], [ 0.39,-0.2 ,-0.04], [ 0.35, 0.09, 0.53], ..., [ 0.48,-0.56,-0.27], [ 0.27,-0.21, 0.59], [ 1.25, 0.72, 1.63]]"
"intercept_ intercept_: float or array of shape (n_targets,)Independent term in the linear model. Set to 0.0 if`fit_intercept = False`.","ndarray[float64](18,)","[13.5 ,13.39,12.9 ,...,12.76,11.47,17.8 ]"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`... versionadded:: 0.24,int,3
rank_ rank_: intRank of matrix `X`. Only available when `X` is dense.,int64,np.int64(3)
"singular_ singular_: array of shape (min(X, y),)Singular values of `X`. Only available when `X` is dense.","ndarray[float64](3,)","[567.39,130.24,109.2 ]"


In [38]:
# Model accuracy
y_pred = lr_hist.predict(X_test_processed)
mae = mean_absolute_error(y_test, y_pred)
print(f'South MAE: {mae:.3f}')
mse = mean_squared_error(y_test, y_pred)
print(f'South MSE: {mse:.3f}')
r2 = r2_score(y_test, y_pred)
print(f'South R²: {r2:.3f}')

South MAE: 0.583
South MSE: 0.669
South R²: 0.709


**Verdict:** As far as moisture history is concerned, errors are smaller on the south side than on the north side. However, that side's R² value is lower, suggesting neither exposure is better at using moisture history to predict future moisture content.

### Environment

In [39]:
lr_env = LinearRegression()

#### North Exposure

In [40]:
# The training-testing split
X_train, X_test, y_train, y_test = train_test_split(X_env_north,
                                                    y_north,
                                                    random_state=42,
                                                    test_size=0.20,
                                                    shuffle=False)

In [41]:
# Data cleaning
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

In [42]:
# Model fitting
lr_env.fit(X_train_processed, y_train)

,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies the convergence criterion of the underlying solver. `tol` isset as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. `tol` is set as `cond` of:func:`scipy.linalg.lstsq` when fitting on dense training data... versionadded:: 1.7.. versionchanged:: 1.9 Now supported on dense data, interpreted as the `cond` parameter.",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary <n_jobs>` for more details.",None
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive. Thisoption is only supported for dense arrays.For a comparison between a linear regression model with positive constraintson the regression coefficients and a linear regression without such constraints,see :ref:`sphx_glr_auto_examples_linear_model_plot_nnls.py`... versionadded:: 0.24",False
Name,Type,Value
"coef_ coef_: array of shape (n_features, ) or (n_targets, n_features)Estimated coefficients for the linear regression problem.If multiple targets are passed during the fit (y 2D), thisis a 2D array of shape (n_targets, n_features), while if onlyone target is passed, this is a 1D array of length n_features.","ndarray[float64](18, 5)","[[-0.27, 0.21,-0.55, 0.69,-0.2 ], [-0.44, 0.42,-0.85, 1.24,-0.72], [-0.38, 0.27,-0.71, 0.89,-0.19], ..., [-0.28, 0.22,-0.79, 0.66, 0.22], [-0.18, 0.32,-0.28, 0.35, 0.03], [-0.87, 0.89,-1.88, 1.99,-0.23]]"
"intercept_ intercept_: float or array of shape (n_targets,)Independent term in the linear model. Set to 0.0 if`fit_intercept = False`.","ndarray[float64](18,)","[13.29,15.16,14.37,...,12.54,10.76,17.88]"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`... versionadded:: 0.24,int,5
rank_ rank_: intRank of matrix `X`. Only available when `X` is dense.,int64,np.int64(5)
"singular_ singular_: array of shape (min(X, y),)Singular values of `X`. Only available when `X` is dense.","ndarray[float64](5,)","[788.21,381.41,213.26,164.37,152.01]"


In [43]:
# Model accuracy
y_pred = lr_env.predict(X_test_processed)
mae = mean_absolute_error(y_test, y_pred)
print(f'North MAE: {mae:.3f}')
mse = mean_squared_error(y_test, y_pred)
print(f'North MSE: {mse:.3f}')
r2 = r2_score(y_test, y_pred)
print(f'North R²: {r2:.3f}')

North MAE: 1.627
North MSE: 4.674
North R²: -0.939


#### South Exposure

In [44]:
# The training-testing split
X_train, X_test, y_train, y_test = train_test_split(X_env_south,
                                                    y_south,
                                                    random_state=42,
                                                    test_size=0.20,
                                                    shuffle=False)

In [45]:
# Data cleaning
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

C:\Users\Michael\anaconda3\Lib\site-packages\sklearn\impute\_base.py:647: UserWarning: Skipping features without any observed values: ['S1-TAEM']. At least one non-missing value is needed for imputation with strategy='mean'.
  warnings.warn(
C:\Users\Michael\anaconda3\Lib\site-packages\sklearn\impute\_base.py:647: UserWarning: Skipping features without any observed values: ['S1-TAEM']. At least one non-missing value is needed for imputation with strategy='mean'.
  warnings.warn(


In [46]:
# Model fitting
lr_env.fit(X_train_processed, y_train)

,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies the convergence criterion of the underlying solver. `tol` isset as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. `tol` is set as `cond` of:func:`scipy.linalg.lstsq` when fitting on dense training data... versionadded:: 1.7.. versionchanged:: 1.9 Now supported on dense data, interpreted as the `cond` parameter.",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary <n_jobs>` for more details.",None
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive. Thisoption is only supported for dense arrays.For a comparison between a linear regression model with positive constraintson the regression coefficients and a linear regression without such constraints,see :ref:`sphx_glr_auto_examples_linear_model_plot_nnls.py`... versionadded:: 0.24",False
Name,Type,Value
"coef_ coef_: array of shape (n_features, ) or (n_targets, n_features)Estimated coefficients for the linear regression problem.If multiple targets are passed during the fit (y 2D), thisis a 2D array of shape (n_targets, n_features), while if onlyone target is passed, this is a 1D array of length n_features.","ndarray[float64](18, 4)","[[-0.2 , 0.29,-0.3 , 0.84], [-0.21, 0.26,-0.31, 0.99], [-0.23, 0.26,-0.32, 0.73], ..., [-0.23, 0.33,-0.37, 1.19], [-0.16, 0.34,-0.2 , 0.49], [-0.7 , 1.18,-1.21, 2.57]]"
"intercept_ intercept_: float or array of shape (n_targets,)Independent term in the linear model. Set to 0.0 if`fit_intercept = False`.","ndarray[float64](18,)","[13.5 ,13.39,12.9 ,...,12.76,11.47,17.8 ]"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`... versionadded:: 0.24,int,4
rank_ rank_: intRank of matrix `X`. Only available when `X` is dense.,int64,np.int64(4)
"singular_ singular_: array of shape (min(X, y),)Singular values of `X`. Only available when `X` is dense.","ndarray[float64](4,)","[601.2 ,327.93,181.67,108.35]"


In [47]:
# Model accuracy
y_pred = lr_env.predict(X_test_processed)
mae = mean_absolute_error(y_test, y_pred)
print(f'South MAE: {mae:.3f}')
mse = mean_squared_error(y_test, y_pred)
print(f'South MSE: {mse:.3f}')
r2 = r2_score(y_test, y_pred)
print(f'South R²: {r2:.3f}')

South MAE: 1.441
South MSE: 4.272
South R²: -0.897


**Verdict:** Negative R² values indicate temperature and humidity (environmental factors) cannot reliably predict moisture content. Perhaps combining them with moisture values (history) will yield better results.

### History and Environment Combined

In [48]:
lr_combined = LinearRegression()

#### North Exposure

In [49]:
# The training-testing split
X_north = pd.concat([X_hist_north, X_env_north], axis=1)
X_train, X_test, y_train, y_test = train_test_split(X_north,
                                                    y_north,
                                                    random_state=42,
                                                    test_size=0.20,
                                                    shuffle=False)

In [50]:
# Data cleaning
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

In [51]:
# Model fitting
lr_combined.fit(X_train_processed, y_train)

,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies the convergence criterion of the underlying solver. `tol` isset as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. `tol` is set as `cond` of:func:`scipy.linalg.lstsq` when fitting on dense training data... versionadded:: 1.7.. versionchanged:: 1.9 Now supported on dense data, interpreted as the `cond` parameter.",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary <n_jobs>` for more details.",None
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive. Thisoption is only supported for dense arrays.For a comparison between a linear regression model with positive constraintson the regression coefficients and a linear regression without such constraints,see :ref:`sphx_glr_auto_examples_linear_model_plot_nnls.py`... versionadded:: 0.24",False
Name,Type,Value
"coef_ coef_: array of shape (n_features, ) or (n_targets, n_features)Estimated coefficients for the linear regression problem.If multiple targets are passed during the fit (y 2D), thisis a 2D array of shape (n_targets, n_features), while if onlyone target is passed, this is a 1D array of length n_features.","ndarray[float64](18, 6)","[[ 0.28, 0.24,-0.12, 0.14,-0.13,-0.01], [ 0.47, 0.42,-0.14, 0.42,-0.52,-0.21], [ 0.39, 0.27,-0.12, 0.14,-0.13,-0. ], ..., [ 0.3 , 0.26,-0.17,-0.24, 0.27, 0.04], [ 0.18, 0.17, 0.16,-0.04, 0.09,-0.01], [ 0.93, 0.83,-0.18, 0.13,-0.28, 0.65]]"
"intercept_ intercept_: float or array of shape (n_targets,)Independent term in the linear model. Set to 0.0 if`fit_intercept = False`.","ndarray[float64](18,)","[13.29,15.16,14.37,...,12.54,10.76,17.88]"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`... versionadded:: 0.24,int,6
rank_ rank_: intRank of matrix `X`. Only available when `X` is dense.,int64,np.int64(6)
"singular_ singular_: array of shape (min(X, y),)Singular values of `X`. Only available when `X` is dense.","ndarray[float64](6,)","[1002.36, 495.14, 358.22, 194.98, 171.24, 138.54]"


In [52]:
# Model accuracy
y_pred = lr_combined.predict(X_test_processed)
mae = mean_absolute_error(y_test, y_pred)
print(f'North MAE: {mae:.3f}')
mse = mean_squared_error(y_test, y_pred)
print(f'North MSE: {mse:.3f}')
r2 = r2_score(y_test, y_pred)
print(f'North R²: {r2:.3f}')

North MAE: 0.624
North MSE: 0.811
North R²: 0.776


#### South Exposure

In [53]:
# The training-testing split
X_south = pd.concat([X_hist_south, X_env_south], axis=1)
X_train, X_test, y_train, y_test = train_test_split(X_south,
                                                    y_south,
                                                    random_state=42,
                                                    test_size=0.20,
                                                    shuffle=False)

In [54]:
# Data cleaning
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

C:\Users\Michael\anaconda3\Lib\site-packages\sklearn\impute\_base.py:647: UserWarning: Skipping features without any observed values: ['S1-TAEM']. At least one non-missing value is needed for imputation with strategy='mean'.
  warnings.warn(
C:\Users\Michael\anaconda3\Lib\site-packages\sklearn\impute\_base.py:647: UserWarning: Skipping features without any observed values: ['S1-TAEM']. At least one non-missing value is needed for imputation with strategy='mean'.
  warnings.warn(


In [55]:
# Model fitting
lr_combined.fit(X_train_processed, y_train)

,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies the convergence criterion of the underlying solver. `tol` isset as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. `tol` is set as `cond` of:func:`scipy.linalg.lstsq` when fitting on dense training data... versionadded:: 1.7.. versionchanged:: 1.9 Now supported on dense data, interpreted as the `cond` parameter.",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary <n_jobs>` for more details.",None
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive. Thisoption is only supported for dense arrays.For a comparison between a linear regression model with positive constraintson the regression coefficients and a linear regression without such constraints,see :ref:`sphx_glr_auto_examples_linear_model_plot_nnls.py`... versionadded:: 0.24",False
Name,Type,Value
"coef_ coef_: array of shape (n_features, ) or (n_targets, n_features)Estimated coefficients for the linear regression problem.If multiple targets are passed during the fit (y 2D), thisis a 2D array of shape (n_targets, n_features), while if onlyone target is passed, this is a 1D array of length n_features.","ndarray[float64](18, 5)","[[ 0.25, 0.2 ,-0.08, 0.02,-0.08], [ 0.27, 0.21,-0.16, 0.1 ,-0.18], [ 0.26, 0.14,-0.04,-0.1 , 0.07], ..., [ 0.32, 0.29,-0.22, 0.23,-0.53], [ 0.19, 0.17, 0.12,-0.03,-0.18], [ 0.88, 0.72,-0.05,-0.46, 0.61]]"
"intercept_ intercept_: float or array of shape (n_targets,)Independent term in the linear model. Set to 0.0 if`fit_intercept = False`.","ndarray[float64](18,)","[13.5 ,13.39,12.9 ,...,12.76,11.47,17.8 ]"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`... versionadded:: 0.24,int,5
rank_ rank_: intRank of matrix `X`. Only available when `X` is dense.,int64,np.int64(5)
"singular_ singular_: array of shape (min(X, y),)Singular values of `X`. Only available when `X` is dense.","ndarray[float64](5,)","[732.31,442.23,280.25,178.09,128.96]"


In [56]:
# Model accuracy
y_pred = lr_combined.predict(X_test_processed)
mae = mean_absolute_error(y_test, y_pred)
print(f'South MAE: {mae:.3f}')
mse = mean_squared_error(y_test, y_pred)
print(f'South MSE: {mse:.3f}')
r2 = r2_score(y_test, y_pred)
print(f'South R²: {r2:.3f}')

South MAE: 0.567
South MSE: 0.588
South R²: 0.705


**Verdict:** The story here is similar to that of moisture history -- errors are lower on the south exposure, but R² is lower as well. This means that even when moisture history and environmental factors are combined, they are no more accurate on one side than another.

## Build Machine Learning Pipelines

### History

In [57]:
hist_pipeline = Pipeline([('preprocessing', preprocessor),
                          ('regr', LinearRegression())])

#### North Exposure

In [58]:
# The training-testing split
X_train, X_test, y_train, y_test = train_test_split(X_hist_north,
                                                    y_north,
                                                    random_state=42,
                                                    test_size=0.20,
                                                    shuffle=False)

In [59]:
# Pipeline fitting
hist_pipeline.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessing', ...), ('regr', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](36,)","['N1-MENU-MC_lag_1','N1-MENM-MC_lag_1','N1-MENL-MC_lag_1',..., 'N3-MDEM-MC_mean_previous_24','N3-MDIM-MC_mean_previous_24', 'N3-MCEM-MC_mean_previous_24']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,36
,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('imputer', ...), ('scale', ...), ...]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimator

##### Hyperparameter Tuning

In [60]:
search_space = [{'regr': [LinearRegression()], 'regr__fit_intercept': [True, False]},
                {'regr': [Ridge(), Lasso()], 'preprocessing__pca__n_components': [0.90, 0.95, 0.99, None]}]
gs = GridSearchCV(hist_pipeline,
                  param_grid=search_space,
                  scoring='neg_mean_squared_error',
                  cv=5,
                  n_jobs=-1)
gs.fit(X_train, y_train)

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...egression())])
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","[{'regr': [LinearRegression()], 'regr__fit_intercept': [True, False]}, {'preprocessing__pca__n_components': [0.9, 0.95, ...], 'regr': [Ridge(), Lasso()]}]"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'neg_mean_squared_error'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",5
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example<sphx_glr_auto_examples_model_selection_plot_grid_search_refit_callable.py>`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchange

In [61]:
best_pipeline = gs.best_estimator_
best_regression_model = best_pipeline.named_steps['regr']
print('Ideal north exposure regression model:', best_regression_model)
print('North exposure hyperparameters:', best_regression_model.get_params())

Ideal north exposure regression model: Ridge()
North exposure hyperparameters: {'alpha': 1.0, 'copy_X': True, 'fit_intercept': True, 'max_iter': None, 'positive': False, 'random_state': None, 'solver': 'auto', 'tol': 0.0001}


In [62]:
y_pred = best_pipeline.predict(X_test)
mae = mean_absolute_error(y_test, y_pred)
print(f'North MAE: {mae:.3f}')
mse = mean_squared_error(y_test, y_pred)
print(f'North MSE: {mse:.3f}')
r2 = r2_score(y_test, y_pred)
print(f'North R²: {r2:.3f}')

North MAE: 0.122
North MSE: 0.035
North R²: 0.986


#### South Exposure

In [63]:
# The training-testing split
X_train, X_test, y_train, y_test = train_test_split(X_hist_south,
                                                    y_south,
                                                    random_state=42,
                                                    test_size=0.20,
                                                    shuffle=False)

In [64]:
# Pipeline fitting
hist_pipeline.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessing', ...), ('regr', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](36,)","['S1-MENU-MC_lag_1','S1-MENM-MC_lag_1','S1-MENL-MC_lag_1',..., 'S3-MDEM-MC_mean_previous_24','S3-MDIM-MC_mean_previous_24', 'S3-MCEM-MC_mean_previous_24']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,36
,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('imputer', ...), ('scale', ...), ...]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimator

##### Hyperparameter Tuning

In [65]:
search_space = [{'regr': [LinearRegression()], 'regr__fit_intercept': [True, False]},
                {'regr': [Ridge(), Lasso()], 'preprocessing__pca__n_components': [0.90, 0.95, 0.99, None]}]
gs = GridSearchCV(hist_pipeline,
                  param_grid=search_space,
                  scoring='neg_mean_squared_error',
                  cv=5,
                  n_jobs=-1)
gs.fit(X_train, y_train)

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...egression())])
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","[{'regr': [LinearRegression()], 'regr__fit_intercept': [True, False]}, {'preprocessing__pca__n_components': [0.9, 0.95, ...], 'regr': [Ridge(), Lasso()]}]"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'neg_mean_squared_error'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",5
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example<sphx_glr_auto_examples_model_selection_plot_grid_search_refit_callable.py>`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchange

In [66]:
best_pipeline = gs.best_estimator_
best_regression_model = best_pipeline.named_steps['regr']
print('Ideal south exposure regression model:', best_regression_model)
print('South exposure hyperparameters:', best_regression_model.get_params())

Ideal south exposure regression model: Ridge()
South exposure hyperparameters: {'alpha': 1.0, 'copy_X': True, 'fit_intercept': True, 'max_iter': None, 'positive': False, 'random_state': None, 'solver': 'auto', 'tol': 0.0001}


In [67]:
y_pred = best_pipeline.predict(X_test)
mae = mean_absolute_error(y_test, y_pred)
print(f'South MAE: {mae:.3f}')
mse = mean_squared_error(y_test, y_pred)
print(f'South MSE: {mse:.3f}')
r2 = r2_score(y_test, y_pred)
print(f'South R²: {r2:.3f}')

South MAE: 0.139
South MSE: 0.046
South R²: 0.966


**Verdict:** Moisture history is a more accurate predictor on the north side than the south side. Not only are errors lower, but R² is higher.

### Environment

In [68]:
env_pipeline = Pipeline([('preprocessing', preprocessor),
                         ('regr', LinearRegression())])

#### North Exposure

In [69]:
# The training-testing split
X_train, X_test, y_train, y_test = train_test_split(X_env_north,
                                                    y_north,
                                                    random_state=42,
                                                    test_size=0.20,
                                                    shuffle=False)

In [70]:
# Pipeline fitting
env_pipeline.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessing', ...), ('regr', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](52,)","['N1-TCEM','N1-TCNM','N1-TCIM',...,'T out','RH out','RH out-DP']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,52
,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('imputer', ...), ('scale', ...), ...]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consum

##### Hyperparameter Tuning

In [71]:
search_space = [{'regr': [LinearRegression()], 'regr__fit_intercept': [True, False]},
                {'regr': [Ridge(), Lasso()], 'preprocessing__pca__n_components': [0.90, 0.95, 0.99, None]}]
gs = GridSearchCV(env_pipeline,
                  param_grid=search_space,
                  scoring='neg_mean_squared_error',
                  cv=5,
                  n_jobs=-1)
gs.fit(X_train, y_train)

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...egression())])
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","[{'regr': [LinearRegression()], 'regr__fit_intercept': [True, False]}, {'preprocessing__pca__n_components': [0.9, 0.95, ...], 'regr': [Ridge(), Lasso()]}]"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'neg_mean_squared_error'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",5
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example<sphx_glr_auto_examples_model_selection_plot_grid_search_refit_callable.py>`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchange

In [72]:
best_pipeline = gs.best_estimator_
best_regression_model = best_pipeline.named_steps['regr']
print('Ideal north exposure regression model:', best_regression_model)
print('North exposure hyperparameters:', best_regression_model.get_params())

Ideal north exposure regression model: Ridge()
North exposure hyperparameters: {'alpha': 1.0, 'copy_X': True, 'fit_intercept': True, 'max_iter': None, 'positive': False, 'random_state': None, 'solver': 'auto', 'tol': 0.0001}


In [73]:
y_pred = best_pipeline.predict(X_test)
mae = mean_absolute_error(y_test, y_pred)
print(f'North MAE: {mae:.3f}')
mse = mean_squared_error(y_test, y_pred)
print(f'North MSE: {mse:.3f}')
r2 = r2_score(y_test, y_pred)
print(f'North R²: {r2:.3f}')

North MAE: 0.924
North MSE: 2.022
North R²: 0.472


#### South Exposure

In [74]:
# The training-testing split
X_train, X_test, y_train, y_test = train_test_split(X_env_south,
                                                    y_south,
                                                    random_state=42,
                                                    test_size=0.20,
                                                    shuffle=False)

In [75]:
# Pipeline fitting
env_pipeline.fit(X_train, y_train)

C:\Users\Michael\anaconda3\Lib\site-packages\sklearn\impute\_base.py:647: UserWarning: Skipping features without any observed values: ['S1-TAEM']. At least one non-missing value is needed for imputation with strategy='mean'.
  warnings.warn(


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessing', ...), ('regr', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](53,)","['S1-TCEM','S1-TCNM','S1-TCIM',...,'T out','RH out','RH out-DP']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,53
,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('imputer', ...), ('scale', ...), ...]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consum

##### Hyperparameter Tuning

In [76]:
search_space = [{'regr': [LinearRegression()], 'regr__fit_intercept': [True, False]},
                {'regr': [Ridge(), Lasso()], 'preprocessing__pca__n_components': [0.90, 0.95, 0.99, None]}]
gs = GridSearchCV(env_pipeline,
                  param_grid=search_space,
                  scoring='neg_mean_squared_error',
                  cv=5,
                  n_jobs=-1)
gs.fit(X_train, y_train)

C:\Users\Michael\anaconda3\Lib\site-packages\sklearn\impute\_base.py:647: UserWarning: Skipping features without any observed values: ['S1-TAEM']. At least one non-missing value is needed for imputation with strategy='mean'.
  warnings.warn(


,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...egression())])
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","[{'regr': [LinearRegression()], 'regr__fit_intercept': [True, False]}, {'preprocessing__pca__n_components': [0.9, 0.95, ...], 'regr': [Ridge(), Lasso()]}]"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'neg_mean_squared_error'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",5
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example<sphx_glr_auto_examples_model_selection_plot_grid_search_refit_callable.py>`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchange

In [77]:
best_pipeline = gs.best_estimator_
best_regression_model = best_pipeline.named_steps['regr']
print('Ideal south exposure regression model:', best_regression_model)
print('South exposure hyperparameters:', best_regression_model.get_params())

Ideal south exposure regression model: Ridge()
South exposure hyperparameters: {'alpha': 1.0, 'copy_X': True, 'fit_intercept': True, 'max_iter': None, 'positive': False, 'random_state': None, 'solver': 'auto', 'tol': 0.0001}


In [78]:
y_pred = best_pipeline.predict(X_test)
mae = mean_absolute_error(y_test, y_pred)
print(f'South MAE: {mae:.3f}')
mse = mean_squared_error(y_test, y_pred)
print(f'South MSE: {mse:.3f}')
r2 = r2_score(y_test, y_pred)
print(f'South R²: {r2:.3f}')

South MAE: 0.587
South MSE: 0.675
South R²: 0.674


C:\Users\Michael\anaconda3\Lib\site-packages\sklearn\impute\_base.py:647: UserWarning: Skipping features without any observed values: ['S1-TAEM']. At least one non-missing value is needed for imputation with strategy='mean'.
  warnings.warn(


**Verdict:** While environmental factors can better predict moisture content on the south side than the north side, said factors do a poorer job in general because errors are higher and R² lower.

### History and Environment Combined

In [87]:
combined_pipeline = Pipeline([('preprocessing', preprocessor),
                              ('regr', LinearRegression())])

#### North Exposure

In [88]:
# The training-testing split
X_north = pd.concat([X_hist_north, X_env_north], axis=1)
X_train, X_test, y_train, y_test = train_test_split(X_north,
                                                    y_north,
                                                    random_state=42,
                                                    test_size=0.20,
                                                    shuffle=False)

In [89]:
# Pipeline fitting
combined_pipeline.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessing', ...), ('regr', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](88,)","['N1-MENU-MC_lag_1','N1-MENM-MC_lag_1','N1-MENL-MC_lag_1',...,'T out', 'RH out','RH out-DP']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,88
,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('imputer', ...), ('scale', ...), ...]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous

##### Hyperparameter Tuning

In [90]:
search_space = [{'regr': [LinearRegression()], 'regr__fit_intercept': [True, False]},
                {'regr': [Ridge(), Lasso()], 'preprocessing__pca__n_components': [0.90, 0.95, 0.99, None]}]
gs = GridSearchCV(combined_pipeline,
                  param_grid=search_space,
                  scoring='neg_mean_squared_error',
                  cv=5,
                  n_jobs=-1)
gs.fit(X_train, y_train)

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...egression())])
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","[{'regr': [LinearRegression()], 'regr__fit_intercept': [True, False]}, {'preprocessing__pca__n_components': [0.9, 0.95, ...], 'regr': [Ridge(), Lasso()]}]"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'neg_mean_squared_error'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",5
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example<sphx_glr_auto_examples_model_selection_plot_grid_search_refit_callable.py>`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchange

In [91]:
best_pipeline = gs.best_estimator_
best_regression_model = best_pipeline.named_steps['regr']
print('Ideal north exposure regression model:', best_regression_model)
print('North exposure hyperparameters:', best_regression_model.get_params())

Ideal north exposure regression model: Ridge()
North exposure hyperparameters: {'alpha': 1.0, 'copy_X': True, 'fit_intercept': True, 'max_iter': None, 'positive': False, 'random_state': None, 'solver': 'auto', 'tol': 0.0001}


In [92]:
y_pred = best_pipeline.predict(X_test)
mae = mean_absolute_error(y_test, y_pred)
print(f'North MAE: {mae:.3f}')
mse = mean_squared_error(y_test, y_pred)
print(f'North MSE: {mse:.3f}')
r2 = r2_score(y_test, y_pred)
print(f'North R²: {r2:.3f}')

North MAE: 0.098
North MSE: 0.021
North R²: 0.989


#### South Exposure

In [93]:
# The training-testing split
X_south = pd.concat([X_hist_south, X_env_south], axis=1)
X_train, X_test, y_train, y_test = train_test_split(X_south,
                                                    y_south,
                                                    random_state=42,
                                                    test_size=0.20,
                                                    shuffle=False)

In [94]:
# Pipeline fitting
combined_pipeline.fit(X_train, y_train)

C:\Users\Michael\anaconda3\Lib\site-packages\sklearn\impute\_base.py:647: UserWarning: Skipping features without any observed values: ['S1-TAEM']. At least one non-missing value is needed for imputation with strategy='mean'.
  warnings.warn(


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessing', ...), ('regr', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](89,)","['S1-MENU-MC_lag_1','S1-MENM-MC_lag_1','S1-MENL-MC_lag_1',...,'T out', 'RH out','RH out-DP']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,89
,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('imputer', ...), ('scale', ...), ...]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous

##### Hyperparameter Tuning

In [95]:
search_space = [{'regr': [LinearRegression()], 'regr__fit_intercept': [True, False]},
                {'regr': [Ridge(), Lasso()], 'preprocessing__pca__n_components': [0.90, 0.95, 0.99, None]}]
gs = GridSearchCV(combined_pipeline,
                  param_grid=search_space,
                  scoring='neg_mean_squared_error',
                  cv=5,
                  n_jobs=-1)
gs.fit(X_train, y_train)

C:\Users\Michael\anaconda3\Lib\site-packages\sklearn\impute\_base.py:647: UserWarning: Skipping features without any observed values: ['S1-TAEM']. At least one non-missing value is needed for imputation with strategy='mean'.
  warnings.warn(


,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...egression())])
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","[{'regr': [LinearRegression()], 'regr__fit_intercept': [True, False]}, {'preprocessing__pca__n_components': [0.9, 0.95, ...], 'regr': [Ridge(), Lasso()]}]"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'neg_mean_squared_error'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",5
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example<sphx_glr_auto_examples_model_selection_plot_grid_search_refit_callable.py>`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchange

In [96]:
best_pipeline = gs.best_estimator_
best_regression_model = best_pipeline.named_steps['regr']
print('Ideal south exposure regression model:', best_regression_model)
print('South exposure hyperparameters:', best_regression_model.get_params())

Ideal south exposure regression model: Ridge()
South exposure hyperparameters: {'alpha': 1.0, 'copy_X': True, 'fit_intercept': True, 'max_iter': None, 'positive': False, 'random_state': None, 'solver': 'auto', 'tol': 0.0001}


In [98]:
y_pred = best_pipeline.predict(X_test)
mae = mean_absolute_error(y_test, y_pred)
print(f'South MAE: {mae:.3f}')
mse = mean_squared_error(y_test, y_pred)
print(f'South MSE: {mse:.3f}')
r2 = r2_score(y_test, y_pred)
print(f'South R²: {r2:.3f}')

South MAE: 0.111
South MSE: 0.026
South R²: 0.973


C:\Users\Michael\anaconda3\Lib\site-packages\sklearn\impute\_base.py:647: UserWarning: Skipping features without any observed values: ['S1-TAEM']. At least one non-missing value is needed for imputation with strategy='mean'.
  warnings.warn(


**Verdict:** When environmental factors and moisture history are combined, moisture content is better predicted on the north side than the south side.